# Semi-compensatory K=3: one-sided interaction

$$\eta_n=\sum_k A_{b_nk}\theta_{m_nk}-D_{b_n}+\sum_{j<k}\gamma_{jk}\,\tilde\theta_{m_nj}\tilde\theta_{m_nk},
\qquad \tilde\theta=\operatorname{softplus}(\theta)>0$$

$$\gamma_{jk}=|z_{jk}|\cdot 0.15,\quad z\sim\text{HalfNormal}(1),
\qquad \mu_n=c_{b_n}+(1-c_{b_n})\,\sigma(\eta_n),
\qquad \text{score}_n\sim\text{Beta}(\mu_n\phi_{b_n},(1-\mu_n)\phi_{b_n})$$

Non-negative loadings, fixed-c 3PL floors, human + lineage $\theta$ priors, pooled $\gamma$.
4 chains x 3000 draws (3000 tune), 758 models / 96 benchmarks / 4274 obs, 62 min, 0 divergences.


In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "fit.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import arviz as az
import numpy as np
import plotly.graph_objects as go
import xarray as xr
from plotly.subplots import make_subplots
from scipy.stats import gaussian_kde, halfnorm, norm

TRACE = ROOT / "results/mirt_interaction/trace_mirt_interaction_gpooled_normal_hp_lp_floors.nc"
PLOTS = ROOT / "plots/interaction_gamma_onesided"
PLOTS.mkdir(parents=True, exist_ok=True)
THIN = 5

C = dict(blue="#0072B2", sky="#56B4E9", orange="#E69F00", verm="#D55E00",
         green="#009E73", pink="#CC79A7", gray="#999999", dark="#333333")


def show(fig, name):
    fig.update_layout(template="plotly_white", title=dict(x=0.5, font=dict(size=13)))
    fig.write_image(PLOTS / f"{name}.png", scale=2)
    fig.show()


post = xr.open_dataset(TRACE, group="posterior")
AXIS = json.loads(post.attrs["mirt_axis_names"])
SCALE = float(post.attrs["mirt_interaction_scale"])
PAIRS = [(0, 1), (0, 2), (1, 2)]
PAIR_NAME = [f"axis {j + 1}×axis {k + 1}" for j, k in PAIRS]
PAIR_COL = [C["blue"], C["orange"], C["green"]]
PRIOR_MED = float(norm.ppf(0.75)) * SCALE          # median of |z| * scale

sub = (post[["gamma_pooled", "A", "theta", "theta_pos", "D", "sigma_b", "tau_CD", "tau_A"]]
       .isel(draw=slice(None, None, THIN)).load())
stats = xr.open_dataset(TRACE, group="sample_stats")[["logp", "diverging"]].load()
N_DRAWS = int(post.sizes["chain"] * post.sizes["draw"])
DIV = int(stats["diverging"].sum())
N_CHAIN = int(post.sizes["chain"])

# Non-negative loadings without founders leave the axis ORDER free, so each chain
# labels the same three axes differently. Everything read per axis or per axis-pair
# is first mapped into chain 0's frame; eta (permutation-invariant) uses raw values.
from itertools import permutations

A = sub["A"].values                                       # (chain, draw, bench, latent)
BENCH = np.array(sub.coords["bench"].values, dtype=object)
A0 = np.median(A[0], axis=0)


def best_perm(c):
    # column permutation of chain c's loadings that best matches chain 0
    Ac = np.median(A[c], axis=0)
    pm = max(permutations(range(3)),
             key=lambda q: sum(np.corrcoef(A0[:, i], Ac[:, q[i]])[0, 1] for i in range(3)))
    return pm, [np.corrcoef(A0[:, i], Ac[:, pm[i]])[0, 1] for i in range(3)]


PERM, CORR = zip(*[best_perm(c) for c in range(N_CHAIN)])
CORR = np.array(CORR)
# chain-0 pair (i,j) sits at this position in chain c's own pair ordering
PAIR_IDX = np.array([[PAIRS.index(tuple(sorted((PERM[c][i], PERM[c][j])))) for i, j in PAIRS]
                     for c in range(N_CHAIN)])
GAM_RAW = sub["gamma_pooled"].values                      # each chain in its own frame
GAM = np.stack([GAM_RAW[c][:, PAIR_IDX[c]] for c in range(N_CHAIN)])   # chain-0 frame

print(f"{N_CHAIN} chains x {int(post.sizes['draw'])} draws | {DIV} divergences / {N_DRAWS} "
      f"| prior median gamma {PRIOR_MED:.4f} | axis perms {[tuple(q) for q in PERM]}")


4 chains x 3000 draws | 0 divergences / 12000 | prior median gamma 0.1012 | axis perms [(0, 1, 2), (2, 1, 0), (2, 0, 1), (0, 2, 1)]


### 1 · Pooled γ: prior vs posterior


In [2]:
g = GAM.reshape(-1, 3)                                         # (S, pair), chain-0 frame
u = np.linspace(-4.0, 0.2, 500)                                # log10 gamma
gx = 10.0 ** u
prior_u = halfnorm.pdf(gx, scale=SCALE) * gx * np.log(10.0)    # density of log10(gamma)

fig = go.Figure()
fig.add_trace(go.Scatter(x=u, y=prior_u, name=f"prior  |z|·{SCALE},  z~HalfNormal(1)",
                         line=dict(color=C["gray"], width=2, dash="dash"),
                         fill="tozeroy", fillcolor="rgba(153,153,153,0.20)"))
for p in range(3):
    k = gaussian_kde(np.log10(g[:, p]))
    fig.add_trace(go.Scatter(x=u, y=k(u), name=f"posterior  {PAIR_NAME[p]}",
                             line=dict(color=PAIR_COL[p], width=2)))
YMAX = 1.15 * max(prior_u.max(), max(gaussian_kde(np.log10(g[:, p]))(u).max() for p in range(3)))
fig.add_trace(go.Scatter(x=[np.log10(PRIOR_MED)] * 2, y=[0, YMAX], mode="lines",
                         name=f"prior median {PRIOR_MED:.3f}",
                         line=dict(color=C["dark"], width=1.5, dash="dot")))
fig.update_layout(
    title="Pooled interaction γ shrinks to 4% of its prior median",
    height=470, width=900,
    xaxis=dict(title="γ  (logits per unit θ̃ⱼθ̃ₖ)", tickvals=[-4, -3, -2, -1, 0],
               ticktext=["0.0001", "0.001", "0.01", "0.1", "1"]),
    yaxis=dict(title="density of log₁₀ γ", range=[0, YMAX]),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5))
show(fig, "p1_gamma_prior_posterior")


### 2 · γ by chain


In [3]:
gc = GAM                                                        # (chain, draw, pair)
OFF = np.linspace(-0.12, 0.12, N_CHAIN)
fig = go.Figure()
for c in range(N_CHAIN):
    med = np.median(gc[c], axis=0)
    q94 = np.quantile(gc[c], 0.94, axis=0)
    fig.add_trace(go.Scatter(
        x=np.arange(3) + OFF[c], y=med, mode="markers", name=f"chain {c}",
        marker=dict(size=10, color=[C["blue"], C["orange"], C["green"], C["pink"]][c],
                    symbol=["circle", "square", "diamond", "triangle-up"][c]),
        error_y=dict(type="data", symmetric=False, array=q94 - med,
                     arrayminus=np.zeros(3), thickness=1.2, width=6)))
fig.add_hline(y=PRIOR_MED, line=dict(color=C["dark"], width=1.5, dash="dot"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", name=f"prior median {PRIOR_MED:.3f}",
                         line=dict(color=C["dark"], width=1.5, dash="dot")))
fig.update_layout(title="Every chain puts γ at the boundary (median, whisker to the 94th percentile)",
                  height=420, width=800,
                  xaxis=dict(tickvals=list(range(3)), ticktext=PAIR_NAME, range=[-0.4, 2.4]),
                  yaxis=dict(title="γ", type="log", range=[-3.2, -0.75],
                             tickvals=[0.001, 0.003, 0.01, 0.03, 0.1]),
                  legend=dict(orientation="h", yanchor="bottom", y=1.02,
                              xanchor="center", x=0.5))
show(fig, "p2_gamma_by_chain")


### 3 · Interaction contribution to η, by ability


In [4]:
tp = np.median(sub["theta_pos"].values.reshape(-1, sub.sizes["model"], 3), axis=0)   # (model, 3)
th = np.median(sub["theta"].values.reshape(-1, sub.sizes["model"], 3), axis=0)
gm = np.median(g, axis=0)                                        # posterior median gamma per pair
A_bar = np.median(sub["A"].values.reshape(-1, sub.sizes["bench"], 3), axis=0).mean(axis=0)

prod = np.stack([tp[:, j] * tp[:, k] for j, k in PAIRS], axis=1)  # (model, pair)
inter_post = (prod * gm).sum(axis=1)
inter_prior = prod.sum(axis=1) * PRIOR_MED
main = th @ A_bar
x = th.mean(axis=1)
o = np.argsort(x)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x[o], y=main[o], mode="lines", name="main effect  Σₖ Āₖθₖ",
                         line=dict(color=C["gray"], width=2)))
fig.add_trace(go.Scatter(x=x[o], y=inter_prior[o], mode="lines",
                         name=f"interaction at prior median γ={PRIOR_MED:.3f}",
                         line=dict(color=C["verm"], width=2, dash="dash")))
fig.add_trace(go.Scatter(x=x[o], y=inter_post[o], mode="lines",
                         name="interaction at posterior median γ",
                         line=dict(color=C["blue"], width=2.5)))
fig.update_layout(
    title=f"The fitted interaction is worth at most {inter_post.max():.2f} logits; "
          f"at the prior median it would reach {inter_prior.max():.2f}",
    height=430, width=800,
    xaxis_title="mean raw ability θ over the three axes",
    yaxis_title="contribution to η (logits)",
    legend=dict(yanchor="top", y=0.98, xanchor="left", x=0.02))
show(fig, "p3_interaction_contribution")


### 4 · r̂: identified quantities vs permutation-inflated ones


In [5]:
rng = np.random.default_rng(0)
n_cell = 400
mi = rng.integers(0, sub.sizes["model"], n_cell)
bi = rng.integers(0, sub.sizes["bench"], n_cell)
Ac, thc, tpc = sub["A"].values[:, :, bi, :], sub["theta"].values[:, :, mi, :], sub["theta_pos"].values[:, :, mi, :]
eta = (Ac * thc).sum(-1) - sub["D"].values[:, :, bi]
for p, (j, k) in enumerate(PAIRS):
    eta = eta + GAM_RAW[:, :, p][:, :, None] * tpc[..., j] * tpc[..., k]
eta_da = xr.Dataset({"eta": xr.DataArray(eta, dims=("chain", "draw", "cell"))})
eta_rhat = float(az.rhat(eta_da)["eta"].max())

rh = az.rhat(sub)
GROUPS = [("η (grid)", eta_rhat, True),
          ("D", float(rh["D"].max()), True),
          ("σ<sub>b</sub>", float(rh["sigma_b"].max()), True),
          ("τ<sub>CD</sub>", float(rh["tau_CD"].max()), True),
          ("τ<sub>A</sub>", float(rh["tau_A"].max()), True),
          ("γ", float(rh["gamma_pooled"].max()), True),
          ("A", float(rh["A"].max()), False),
          ("θ", float(rh["theta"].max()), False),
          ("softplus θ", float(rh["theta_pos"].max()), False)]

fig = go.Figure()
for lab, first in (("identified", True), ("axis-permutation free", False)):
    sel = [(n, v) for n, v, ident in GROUPS if ident is first]
    fig.add_trace(go.Bar(x=[n for n, _ in sel], y=[v for _, v in sel], name=lab,
                         marker_color=C["blue"] if first else C["verm"],
                         text=[f"{v:.3f}" for _, v in sel], textposition="outside"))
fig.add_hline(y=1.01, line=dict(color=C["dark"], width=1, dash="dot"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", name="r̂ = 1.01",
                         line=dict(color=C["dark"], width=1, dash="dot")))
fig.update_layout(title=f"Max r̂ by parameter, {DIV} divergences / {N_DRAWS}",
                  height=440, width=830, yaxis=dict(title="max r̂", range=[0.99, 2.75]),
                  xaxis=dict(tickangle=0),
                  legend=dict(orientation="h", yanchor="bottom", y=1.03,
                              xanchor="center", x=0.5))
show(fig, "p4_rhat_groups")


### 5 · Chains agree on the axes up to their labels


In [6]:
fig = go.Figure(go.Heatmap(
    z=CORR, x=[f"axis {k + 1}" for k in range(3)],
    y=[f"chain {c}  (axes {','.join(str(q + 1) for q in PERM[c])})" for c in range(N_CHAIN)],
    zmin=0.9, zmax=1.0, colorscale=[[0, "#FFF7EC"], [0.5, C["orange"]], [1.0, C["blue"]]],
    colorbar=dict(title="corr", thickness=12), texttemplate="%{z:.3f}",
    textfont=dict(size=12)))
fig.update_layout(
    title="Loading correlation with chain 0 after the best axis permutation",
    height=340, width=760, yaxis=dict(autorange="reversed"))
show(fig, "p5_chain_axis_alignment")


### 6 · What each axis loads on


In [7]:
A_al = np.concatenate([A[c][:, :, list(PERM[c])] for c in range(N_CHAIN)], axis=0)  # (S, bench, 3)
med = np.median(A_al, axis=0)
lo, hi = np.quantile(A_al, [0.03, 0.97], axis=0)
TOPN = 12

fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.19,
                    subplot_titles=[f"axis {k + 1}" for k in range(3)])
for k in range(3):
    idx = np.argsort(-med[:, k])[:TOPN][::-1]
    fig.add_trace(go.Bar(
        y=BENCH[idx], x=med[idx, k], orientation="h", showlegend=False,
        marker_color=PAIR_COL[k],
        error_x=dict(type="data", symmetric=False, array=hi[idx, k] - med[idx, k],
                     arrayminus=med[idx, k] - lo[idx, k], thickness=1, width=3,
                     color=C["dark"])), row=1, col=k + 1)
fig.update_xaxes(title="loading A", range=[0, 1.06 * hi.max()])
fig.update_yaxes(tickfont=dict(size=9))
fig.update_layout(
    title=f"Top {TOPN} loadings per axis, aligned across chains (median, 94% interval)",
    height=520, width=1150, bargap=0.28)
show(fig, "p6_top_loadings")
